# AASHTO MBE - Database Schema Mapping

Original database schema designs mapping AASHTO Manual for Bridge
Evaluation (MBE) Section 2 requirements to SNBI/PostGIS table structures
with compliance logic and validation rules.

Section organization references:

- AASHTO. *The Manual for Bridge Evaluation*, 3rd Ed.
- FHWA. *Specification for the National Bridge Inventory (SNBI)*.

Refer to the current AASHTO MBE for authoritative inspection and
evaluation procedures.


## 2.1 - Introduction

The primary goal of maintaining bridge files is to ensure public safety by  
collecting and organizing accurate, up-to-date, and comprehensive  
information for every in-service highway bridge. These files provide a  
cumulative history of the structure that is essential for engineers to  
review prior to conducting inspections, ratings, or evaluations. The  
documentation may be stored electronically, on paper, or in external  
documents that are referenced within the specific bridge file.

### 2.2.1 - General File Information

This section establishes the identity of the bridge. In the new SNBI schema,  
this is primarily housed in the bridges_bridge table, which uses SNBI  
coding (e.g., bid01, bl05). Visuals and plans are linked via the records  
tables.

- `SNBIInventory` Object
    - Source Table: bridges_bridge
    - Key Fields:
        - `sfn` (Structure File Number): `bid01_bridge_number` (VARCHAR)
        - `location`: `bl11_bridge_location` (VARCHAR)
        - `coordinates`: `location` (PostGIS Geometry/Point) or  
          `bl05_latitude`/`bl06_longitude` (FLOAT)
        - `year_built`: `bw01_year_built` (INT)
        - `nbis_length`: `bg01_nbis_len` (FLOAT)
    - **Compliance Logic**: Validate that bid01 is unique and location  
    geometry is valid WKT (Well-Known Text).
- `BridgeVisuals` Object
    - **Source Table**: records_projectarchive joined with `records_bridgeprojectlink`
    - **Key Fields:**
        - `file_path`: `full_path` (VARCHAR)
        - `file_type`: Derived from `full_path` extension (PDF/JPG)
        - `link_verified`: `is_verified` (BOOL)
- **Compliance Logic**: Query `records_bridgeprojectlink` for the SFN to  
ensure at least one "General Plan" and required photos (Top/Elevation)
exist in the archive.

### **2.2.2 Field Inspection Information**
This section manages the chronological record of bridge conditions. The schema separates the *inspection event* (`bridges_inspection`) from the *element-level data* (`bridges_element`) and the *component ratings* (`bridges_bridge`).

* **`SNBIInspectionEvent` Object**
    * **Source Table:** `bridges_inspection`
    * **Key Fields:**
        * `inspection_date`: `bie02_begin_date` (DATE) 
        * `type`: `bie01_type` (VARCHAR) 
        * `team_leader`: `bie04_inspector_id` (FK to `bridges_inspector`) 
        * `notes`: `bie11_note` (TEXT) - *Captures "observations and findings"* 
        * `qc_date`: `bie08_qc_date` (DATE)
        * `next_due`: `bie06_due_date` (DATE)
    * **Compliance Logic:** Validate that every `bie02_begin_date` has a corresponding PDF report in `records_projectarchive` linked via `records_bridgeprojectlink`.

* **`ComponentCondition` Object**
    * **Source Table:** `bridges_bridge`
    * **Key Fields:**
        * `deck`: `bc01_deck_cond` (VARCHAR) 
        * `superstructure`: `bc02_super_cond` (VARCHAR) 
        * `substructure`: `bc03_sub_cond` (VARCHAR) 
        * `culvert`: `bc04_culvert_cond` (VARCHAR) 
    * **Compliance Logic:** If any rating is ≤ 5, ensure `bie11_note` contains a narrative justification.

* **`ElementDetails` Object**
    * **Source Table:** `bridges_element`
    * **Key Fields:**
        * `element_id`: `be01_number` (VARCHAR)
        * `total_quantity`: `be03_total_qty` (FLOAT)
        * `condition_states`: `bcs01_qty_state_1` through `bcs04_qty_state_4` (FLOAT) 
    * **Compliance Logic:** Verify `bcs03` (CS3) or `bcs04` (CS4) quantities > 0 trigger a specific note or sketch requirement.


### **2.2.3 Critical Findings and Actions Taken**
Critical findings require detailed documentation and immediate action tracking. The schema handles this via condition flags and file links.

* **`CriticalFindingAlert` Object**
    * **Source Table:** `bridges_bridge`
    * **Key Fields:**
        * `nstm_status`: `bc14_nstm_cond` (VARCHAR) - *Primary indicator for NSTM/FC issues* 
        * `scour_status`: `bc11_scour_cond` (VARCHAR)
        * `lowest_rating`: `bc13_lowest_rating` (VARCHAR)
    * **Source Table:** `bridges_inspection`
    * **Key Fields:**
        * `finding_description`: `bie11_note` (TEXT) 
    * **Compliance Logic:** If critical indicators are present, query `records_projectarchive` for files tagged with "Critical Finding" or "Repair" to verify action documentation.

### **2.2.4 Waterway Information**
This section tracks hydraulic data for bridges over water.

* **`HydraulicProfile` Object**
    * **Source Table:** `bridges_feature`
    * **Key Fields:**
        * `feature_type`: `bf01_type` (VARCHAR) - *Used to filter for waterway bridges*
        * `navigable`: `bn01_navigable_waterway` (VARCHAR)
        * `channel_protection`: `bn06_substruct_nav_prot` (VARCHAR)
    * **Source Table:** `bridges_bridge`
    * **Key Fields:**
        * `scour_vulnerability`: `bap03_scour_vuln` (VARCHAR) 
        * `channel_condition`: `bc09_channel_cond` (VARCHAR) 
    * **Compliance Logic:** If `bap03_scour_vuln` indicates a scour-critical bridge, check `records_projectarchive` for "Channel Profile" or "Cross Section" files.

### **2.2.5 Significant Correspondence**
Tracks legal and administrative agreements regarding the bridge.

* **`AdministrativeRecord` Object**
    * **Source Table:** `bridges_bridge`
    * **Key Fields:**
        * `owner`: `bcl01_owner` (VARCHAR) 
        * `maintenance_resp`: `bcl02_maint_resp` (VARCHAR) 
        * `border_agreement`: `bl09_border_insp_resp` (VARCHAR) - *Crucial for border bridges* 
    * **Compliance Logic:** If `bcl01_owner` != `bcl02_maint_resp` (Owner differs from Maintainer), verify an "Agreement" document exists in `records_projectarchive`.

### **2.2.6 Other Inspection Procedures or Requirements**
Specific procedures for complex or hazardous inspections.

* **`SpecializedProcedure` Object**
    * **Source Table:** `bridges_bridge`
    * **Key Fields:**
        * `nstm_required`: `bir01_nstm_req` (VARCHAR) - *Formerly Fracture Critical* 
        * `underwater_required`: `bir03_underwater_req` (VARCHAR) 
        * `complex_feature`: `bir04_complex_feature` (VARCHAR) 
        * `fatigue_prones`: `bir02_fatigue` (VARCHAR)
    * **Compliance Logic:** If any `_req` field is affirmative (e.g., "Y" or a designated interval), ensure a specific "Inspection Procedure" document is linked in `records_projectarchive`.

### **2.2.7 Load Rating Documentation**
Derived data verifying the safe load capacity.

* **`LoadRatingRecord` Object**
    * **Source Table:** `bridges_bridge`
    * **Key Fields:**
        * `design_load_type`: `blr01_design_load` (VARCHAR) 
        * `design_method`: `blr02_design_method` (VARCHAR)
        * `rating_date`: `blr03_load_date` (VARCHAR) 
        * `rating_method`: `blr04_rating_method` (VARCHAR) 
        * `inventory_factor`: `blr05_inv_factor` (FLOAT)
        * `operating_factor`: `blr06_opr_factor` (FLOAT)
    * **Source Table:** `ohio_bridge`
    * **Key Fields:**
        * `software_used`: `ohio_708_rating_software` (VARCHAR) 
    * **Compliance Logic:** Verify `blr03_load_date` is populated. If `blr06_opr_factor` < 1.0, flag for Posting Check.

### **2.2.8 Posting Documentation**
Manages the status of bridges that cannot carry legal loads.

* **`PostingStatus` Object**
    * **Source Table:** `bridges_postingstatus`
    * **Key Fields:**
        * `current_status`: `bps01_status` (VARCHAR) 
        * `date_implemented`: `bps02_change_date` (DATE) 
    * **Source Table:** `bridges_postingevaluation`
    * **Key Fields:**
        * `restriction_type`: `bep03_posting_type` (VARCHAR)
        * `load_limit`: `bep04_posting_value` (VARCHAR)
    * **Compliance Logic:** If `bridges_bridge.blr06_opr_factor` < 1.0, `bps01_status` must equal "PSTD" (Posted) or "CLSD" (Closed). Verify `records_projectarchive` contains a photo of the posting sign.m

### **2.2.9 Scour Assessment**
This section tracks the evaluation of bridge stability during flood events. The schema links the assessment status on the bridge record to the physical characteristics of the feature intersected.

* **`ScourEvaluation` Object**
    * **Source Table:** `bridges_bridge`
    * **Key Fields:**
        * `vulnerability_status`: `bap03_scour_vuln` (VARCHAR) - *Corresponds to NBI Item 113*
        * `scour_condition`: `bc11_scour_cond` (VARCHAR)
    * **Source Table:** `bridges_feature`
    * **Key Fields:**
        * `feature_type`: `bf01_type` (VARCHAR)
    * **Compliance Logic:** If `bridges_feature.bf01_type` indicates a waterway (e.g., "River", "Stream"), `bap03_scour_vuln` must be populated (NOT NULL). If `bap03_scour_vuln` indicates the bridge has been assessed (i.e., not "U" or "T"), a "Scour Assessment" report must exist in `records_projectarchive`.

### **2.2.10 Scour Plan of Action**
For bridges determined to be scour-critical, a specific management plan is mandatory.

* **`POACompliance` Object**
    * **Source Table:** `bridges_bridge`
    * **Key Fields:**
        * `poa_documented`: `bap04_scour_poa` (VARCHAR)
        * `scour_critical_code`: `bap03_scour_vuln` (VARCHAR)
    * **Compliance Logic:** If `bap03_scour_vuln` identifies the bridge as Scour Critical (e.g., codes 0, 1, 2, 3, or U depending on state definition), `bap04_scour_poa` must indicate that a POA is implemented. Query `records_projectarchive` for a file tagged "POA" or "Plan of Action" linked to the specific `bridge_id`.

### **2.3.1 General**
This section serves as a catch-all for supplemental data that supports the Bridge Management System (BMS) but isn't strictly required by NBIS. In your schema, this often relates to specific "AssetWise" fields or archived project data.

* **`SupplementalArchive` Object**
    * **Source Table:** `records_projectarchive`
    * **Key Fields:**
        * `archive_path`: `full_path` (VARCHAR) - *Location of supplemental files*
        * `has_pdf`: `has_pdf` (BOOL)
        * `last_update`: `last_scanned` (TIMESTAMP)
    * **Compliance Logic:** Verify that older bridges (where `year_built` < 1990) have at least one entry in `records_projectarchive` to ensure historical data is being retained.

### **2.3.2 Plans and Drawings**
Essential for load rating and rehabilitation. This tracks the "As-Built" or "Shop" drawings.

* **`AsBuiltRecord` Object**
    * **Source Table:** `records_bridgeprojectlink`
    * **Key Fields:**
        * `link_verified`: `is_verified` (BOOL)
        * `project_date`: `created_at` (TIMESTAMP)
    * **Source Table:** `records_projectarchive`
    * **Key Fields:**
        * `file_count`: `file_count` (INT)
    * **Compliance Logic:** Check for the existence of files tagged "Plans", "As-Built", or "Shop Drawings" linked to the bridge ID.



### **2.3.3 Construction Documentation**
Tracks material quality and construction history, crucial for determining material strengths in load rating. Include any reference to material certifications and tests performed during construction activities such as pile driving, concrete placement, and prestressing operations.

Retain all pertinent certificates for the type, grade and quality of materials in the bridge, like steel mill certificates, concrete delivery slips, and other manufacturers' certifications.

* **`MaterialCertification` Object**
    * **Source Table:** `bridges_spanset`
    * **Key Fields:**
        * `material_type`: `bsp04_material` (VARCHAR)
        * `deck_material`: `bsp09_deck_material` (VARCHAR)
        * `reinforcing_type`: `bsp12_deck_reinforcing` (VARCHAR)
    * **Source Table:** `ohio_borings` (as a proxy for geotechnical construction data)
    * **Key Fields:**
        * `boring_log`: `log_url` (VARCHAR)
        * `boring_data`: `data_url` (VARCHAR)
    * **Compliance Logic:** If `bsp04_material` indicates "Prestressed Concrete" or "Steel", query `records_projectarchive` for "Mill Certs" or "Pile Driving Logs".

### **2.3.4 Original Design Documentation**
This section preserves the original engineering logic, specifically the design standards and calculations.


* **`DesignCriteria` Object**
    * **Source Table:** `bridges_bridge`
    * **Key Fields:**
        * `design_load`: `blr01_design_load` (VARCHAR) - *e.g., "HS-20", "HL-93"*
        * `design_method`: `blr02_design_method` (VARCHAR) - *e.g., "LFD", "ASD", "LRFD"*
    * **Compliance Logic:** Verify that a "Design Calculation" file exists. If `blr02_design_method` is "LRFD", ensure the file references the specific AASHTO spec version used.

### **2.3.5 Unique Considerations**
Handles special access needs, safety hazards, or environmental factors.

Also mentions things like coast guard, security, operations, other agencies times of the year it's best to access, etc.

* **`SpecialAccess` Object**
    * **Source Table:** `bridges_inspection`
    * **Key Fields:**
        * `access_equipment`: `bie12_equipment` (VARCHAR) - *e.g., "Snooper", "Boat"*
    * **Source Table:** `bridges_bridge`
    * **Key Fields:**
        * `fatigue_details`: `bir02_fatigue` (VARCHAR)
        * `complex_features`: `bir04_complex_feature` (VARCHAR)
    * **Compliance Logic:** If `bie12_equipment` lists specialized access (e.g., "UBIT"), verify a "Safety Plan" or "Access Procedure" document is stored.



### **2.3.6 Utilities and Ancillary Attachments**
Tracks non-bridge infrastructure attached to the structure.

* **`UtilityAttachment` Object**
    * **Source Table:** `bridges_assetwisefield` (Hypothetical mapping, as utilities are often custom fields)
    * **Key Fields:**
        * `utility_type`: `fe_name` (VARCHAR) - *Filtered for "Gas", "Electric", "Sewer"*
    * **Compliance Logic:** If utilities are present, ensuring `records_projectarchive` contains a "Utility Agreement" or "Permit" document.



### **2.3.7 Maintenance and Repair History**
A chronological log of work done on the bridge.

* **`WorkHistory` Object**
    * **Source Table:** `bridges_work`
    * **Key Fields:**
        * `work_year`: `bw02_year` (INT)
        * `description`: `bw03_description` (VARCHAR)
    * **Source Table:** `bridges_spanset`
    * **Key Fields:**
        * `protective_system`: `bsp07_protective_system` (VARCHAR)
    * **Compliance Logic:** Create a timeline of `bridges_work` entries. If `bsp07_protective_system` indicates a coating (e.g., "Paint"), verify the last painting date is recorded in `bridges_work`.

### **2.3.8 Additional Waterway Documentation**
Expands on section 2.2.4 with historical data like high-water marks.

* **`HydraulicHistory` Object**
    * **Source Table:** `bridges_bridge`
    * **Key Fields:**
        * `scour_history`: `bap03_scour_vuln` (VARCHAR)
    * **Source Table:** `bridges_inspection`
    * **Key Fields:**
        * `channel_notes`: `bie11_note` (TEXT)
    * **Compliance Logic:** If the bridge is over water, check for "High Water Mark" or "Debris" references in `bie11_note` and link to `stream_plan_file` in the archives.

### **2.3.9 Traffic Data**
Tracks volume and vehicle mix, critical for fatigue analysis and load rating.

* **`TrafficProfile` Object**
    * **Source Table:** `bridges_feature`
    * **Key Fields:**
        * `aadt`: `bh09_aadt` (INT)
        * `truck_percent`: `bh10_aadt_truck` (INT)
        * `traffic_year`: `bh11_year_aadt` (INT)
    * **Compliance Logic:** Ensure `bh11_year_aadt` is within the last 3-5 years (or per ODOT policy). If `bh10_aadt_truck` is high (>20%), verify "Fatigue Analysis" is present in `records_projectarchive`.